# Bank of America Data Analytics Project

## Project Objectives

This project aims to establish a comprehensive data analytics infrastructure for Bank of America using the medallion architecture pattern. The primary objectives include:

* **Customer Insights & Segmentation** - Analyze customer behavior patterns, transaction trends, and account activities to identify high-value segments and personalization opportunities
* **Transaction Monitoring & Analysis** - Track and analyze transaction volumes, patterns, and anomalies across different channels (ATM, online, mobile, branch)
* **Risk Assessment & Fraud Detection** - Identify suspicious activities, unusual transaction patterns, and potential fraud indicators through data-driven analysis
* **Operational Efficiency** - Monitor branch performance, service quality metrics, and resource utilization to optimize operations
* **Regulatory Compliance** - Ensure data governance, audit trails, and reporting capabilities meet regulatory requirements (e.g., AML, KYC, Dodd-Frank)
* **Business Intelligence & Reporting** - Provide executive dashboards and reports for strategic decision-making

## Problem Statement

Bank of America handles millions of daily transactions across diverse products (checking, savings, credit cards, loans, investments) and channels. The current challenges include:

1. **Data Fragmentation** - Customer and transaction data exists in multiple systems with inconsistent formats and quality
2. **Limited Real-Time Insights** - Delays in data processing hinder timely decision-making for fraud prevention and customer service
3. **Scalability Concerns** - Growing data volumes require a robust, scalable architecture to handle petabyte-scale datasets
4. **Compliance Complexity** - Regulatory requirements demand comprehensive audit trails, data lineage, and secure access controls
5. **Analytical Barriers** - Business users need simplified access to trusted, curated datasets without navigating complex raw data structures

## Background

### Data Sources

The project will integrate data from multiple source systems:

* **Core Banking Systems** - Account information, customer profiles, product holdings
* **Transaction Processing Systems** - ATM transactions, online banking, mobile app, wire transfers, ACH payments
* **Credit Card Systems** - Card transactions, authorization data, merchant information
* **Customer Relationship Management (CRM)** - Customer interactions, service requests, complaints
* **Risk & Compliance Systems** - Credit scores, fraud alerts, AML flags, regulatory filings
* **External Data** - Credit bureaus, market data, economic indicators

### Medallion Architecture Implementation

The project follows the medallion architecture pattern with three distinct layers:

**Bronze Layer (Raw Data)**
- Ingests raw data from source systems with minimal transformation
- Preserves original data formats and structures for audit and compliance
- Includes full history and change data capture (CDC) for tracking
- Serves as the single source of truth

**Silver Layer (Cleaned & Transformed)**
- Cleanses and standardizes data (e.g., consistent date formats, currency conversions)
- Deduplicates records and resolves data quality issues
- Enriches data with calculated fields and derived attributes
- Implements business rules and data validation
- Provides queryable datasets for analysts

**Gold Layer (Business-Level Aggregates)**
- Creates pre-aggregated, business-ready datasets optimized for reporting
- Implements key performance indicators (KPIs) and metrics
- Builds customer 360 views, product summaries, and analytical marts
- Optimized for dashboards, BI tools, and executive reporting
- Enforces access controls based on business roles

### Expected Outcomes

* Reduced time-to-insight from days to minutes through optimized data pipelines
* Improved data quality and consistency across the organization
* Enhanced fraud detection capabilities with near real-time monitoring
* Simplified access to trusted data for business users and data scientists
* Scalable foundation supporting future advanced analytics and machine learning initiatives
* Full compliance with regulatory requirements and audit readiness

### Stakeholders

* **Business Users** - Branch managers, product teams, marketing, customer service
* **Data & Analytics Teams** - Data engineers, data scientists, business analysts
* **Risk & Compliance** - Fraud prevention, AML officers, compliance officers
* **Technology Teams** - IT infrastructure, security, architecture
* **Executive Leadership** - C-suite requiring strategic insights and KPI dashboards

---

# Data Quality Rules and Standards

## Overview

Data quality is critical for regulatory compliance, accurate analytics, and operational decision-making.

---

## Bronze Layer - Data Quality Rules

### Purpose
Capture and preserve raw data with minimal validation.

### Key Rules

**1. Completeness Rules**
* All source records must be captured
* Metadata fields required: source_system, ingestion_timestamp, file_name, record_id
* Track rejected records in separate error tables

**2. Technical Validation**
* File format integrity checks
* Schema conformance to expected structure
* No truncated or corrupted records

**3. Audit Trail**
* Preserve original data format and values
* Capture CDC operations
* Maintain full history with effective dates

---

## Silver Layer - Data Quality Rules

### Customer Data Rules

**Customer Identification**
* customer_id - NOT NULL, UNIQUE
* ssn - NULL allowed, when present: 9 digits, encrypted
* date_of_birth - NOT NULL, age between 18-120 years
* email - Valid email format when present

**Customer Profile**
* first_name, last_name - NOT NULL
* address_line1 - NOT NULL for primary address
* city, state, zip_code - NOT NULL, valid US postal codes
* customer_status - IN (ACTIVE, INACTIVE, SUSPENDED, CLOSED)

**KYC and AML Compliance**
* kyc_status - IN (VERIFIED, PENDING, FAILED, EXPIRED)
* risk_rating - IN (LOW, MEDIUM, HIGH, CRITICAL)
* aml_check_date - Must be within last 365 days

### Transaction Data Rules

**Transaction Identification**
* transaction_id - NOT NULL, UNIQUE
* account_id - NOT NULL, must exist in accounts table
* transaction_date - NOT NULL, cannot be future date

**Transaction Amounts**
* amount - NOT NULL, DECIMAL(18,2)
* balance_after greater than or equal to 0
* Currency code - NOT NULL, ISO 4217 format

**Transaction Types**
* transaction_type - IN (DEPOSIT, WITHDRAWAL, TRANSFER, PAYMENT, FEE, INTEREST)
* channel - IN (ATM, ONLINE, MOBILE, BRANCH, ACH, WIRE)
* status - IN (COMPLETED, PENDING, FAILED, REVERSED)

**Fraud Detection Flags**
* Flag transactions greater than ten thousand dollars
* Flag multiple transactions totaling over ten thousand within 24 hours
* Flag transactions in unusual locations
* Flag rapid succession of transactions

### Account Data Rules

**Account Identification**
* account_id - NOT NULL, UNIQUE
* customer_id - NOT NULL, must exist in customers table
* account_number - NOT NULL, UNIQUE, encrypted

**Account Details**
* account_type - IN (CHECKING, SAVINGS, CREDIT_CARD, LOAN, MORTGAGE)
* account_status - IN (ACTIVE, DORMANT, FROZEN, CLOSED)
* current_balance - NOT NULL, DECIMAL(18,2)

### Referential Integrity

* All customer_id references must exist in customers table
* All account_id references must exist in accounts table
* No transactions without valid account
* No accounts without valid customer

---

## Gold Layer - Data Quality Rules

**Aggregation Accuracy**
* Sum of transactions must reconcile with account balances
* Daily transaction counts must match source system reports

**Business Logic Validation**
* Customer lifetime value calculations consistently applied
* Churn rates calculated using standard 90-day window
* Risk scores normalized to 0-1000 range

**Data Freshness**
* Real-time dashboards updated every 5 minutes
* Daily reports available by 6 AM EST
* Monthly reports finalized within 3 business days

---

## Cross-Layer Quality Checks

**Data Consistency**
* Customer counts consistent across Bronze, Silver, Gold
* Transaction volumes reconcile across layers

**Date and Time Standardization**
* All timestamps in UTC
* Date formats consistent: YYYY-MM-DD

**Duplicate Prevention**
* Primary keys enforced at all layers
* Deduplication logic documented

---

## Implementation Approach

**Bronze Layer**: Soft validation - log issues but ingest all data  
**Silver Layer**: Hard validation - reject records failing critical rules  
**Gold Layer**: Business validation - alert on anomalies

**Monitoring**:
* Automated data quality dashboards
* Daily quality score reports
* Alert thresholds for quality degradation

**Remediation**:
* Automated fixes for known patterns
* Manual review queue for ambiguous cases
* Quarterly data quality review meetings

In [0]:
-- Example Data Quality Validation Queries
-- These queries can be used to monitor and enforce data quality rules

-- ============================================
-- CUSTOMER DATA QUALITY CHECKS
-- ============================================

-- Check for missing required customer fields
-- SELECT 
--   COUNT(*) as invalid_records,
--   'Missing required fields' as issue_type
-- FROM Bank_of_America.Bank_of_America_Silver.customers
-- WHERE customer_id IS NULL 
--    OR first_name IS NULL 
--    OR last_name IS NULL 
--    OR date_of_birth IS NULL;

-- Check for invalid customer ages
-- SELECT 
--   customer_id,
--   date_of_birth,
--   DATEDIFF(YEAR, date_of_birth, CURRENT_DATE()) as age
-- FROM Bank_of_America.Bank_of_America_Silver.customers
-- WHERE DATEDIFF(YEAR, date_of_birth, CURRENT_DATE()) < 18 
--    OR DATEDIFF(YEAR, date_of_birth, CURRENT_DATE()) > 120;

-- Check for duplicate customer_ids
-- SELECT 
--   customer_id,
--   COUNT(*) as duplicate_count
-- FROM Bank_of_America.Bank_of_America_Silver.customers
-- GROUP BY customer_id
-- HAVING COUNT(*) > 1;

-- ============================================
-- TRANSACTION DATA QUALITY CHECKS
-- ============================================

-- Check for transactions with invalid amounts
-- SELECT 
--   transaction_id,
--   amount,
--   transaction_type
-- FROM Bank_of_America.Bank_of_America_Silver.transactions
-- WHERE amount IS NULL OR amount <= 0;

-- Check for future-dated transactions
-- SELECT 
--   transaction_id,
--   transaction_date,
--   CURRENT_DATE() as today
-- FROM Bank_of_America.Bank_of_America_Silver.transactions
-- WHERE transaction_date > CURRENT_DATE();

-- Check for orphaned transactions (no matching account)
-- SELECT 
--   t.transaction_id,
--   t.account_id
-- FROM Bank_of_America.Bank_of_America_Silver.transactions t
-- LEFT JOIN Bank_of_America.Bank_of_America_Silver.accounts a
--   ON t.account_id = a.account_id
-- WHERE a.account_id IS NULL;

-- Check for high-value transactions requiring review
-- SELECT 
--   transaction_id,
--   account_id,
--   amount,
--   transaction_date,
--   'High value transaction' as flag_reason
-- FROM Bank_of_America.Bank_of_America_Silver.transactions
-- WHERE amount > 10000;

-- ============================================
-- ACCOUNT DATA QUALITY CHECKS
-- ============================================

-- Check for orphaned accounts (no matching customer)
-- SELECT 
--   a.account_id,
--   a.customer_id
-- FROM Bank_of_America.Bank_of_America_Silver.accounts a
-- LEFT JOIN Bank_of_America.Bank_of_America_Silver.customers c
--   ON a.customer_id = c.customer_id
-- WHERE c.customer_id IS NULL;

-- Check for accounts with invalid close dates
-- SELECT 
--   account_id,
--   open_date,
--   close_date
-- FROM Bank_of_America.Bank_of_America_Silver.accounts
-- WHERE close_date IS NOT NULL 
--   AND close_date < open_date;

-- ============================================
-- CROSS-LAYER RECONCILIATION
-- ============================================

-- Compare record counts between Bronze and Silver layers
-- SELECT 
--   'Bronze' as layer,
--   COUNT(*) as record_count
-- FROM Bank_of_America.Bank_of_America_Bronze.customers_raw
-- UNION ALL
-- SELECT 
--   'Silver' as layer,
--   COUNT(*) as record_count
-- FROM Bank_of_America.Bank_of_America_Silver.customers;

-- ============================================
-- DATA QUALITY SUMMARY DASHBOARD
-- ============================================

-- Comprehensive data quality metrics
-- SELECT 
--   'Customers' as table_name,
--   COUNT(*) as total_records,
--   SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) as null_ids,
--   SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) as missing_emails,
--   CURRENT_TIMESTAMP() as check_timestamp
-- FROM Bank_of_America.Bank_of_America_Silver.customers;

In [0]:
-- Create the Bank of America catalog
CREATE CATALOG IF NOT EXISTS Bank_of_America;

-- Create the Bronze schema for medallion architecture (raw data layer)
CREATE SCHEMA IF NOT EXISTS Bank_of_America.Bank_of_America_Bronze;

-- Create the Silver schema for medallion architecture (cleaned/transformed data layer)
CREATE SCHEMA IF NOT EXISTS Bank_of_America.Bank_of_America_Silver;

-- Create the Gold schema for medallion architecture (business-level aggregates layer)
CREATE SCHEMA IF NOT EXISTS Bank_of_America.Bank_of_America_Gold;

-- Verify creation
SHOW SCHEMAS IN Bank_of_America;

In [0]:
-- ============================================
-- BRONZE LAYER - EXISTING DATA
-- ============================================
-- The Bronze layer already contains your raw data:
-- - consumer_complaints (existing table with consumer complaint data)
-- - consumer_complaints_data_dictionary (data dictionary)

-- Verify Bronze layer tables
SHOW TABLES IN Bank_of_America.Bank_of_America_Bronze;

-- Check record count
SELECT 
  'consumer_complaints' as table_name,
  COUNT(*) as record_count
FROM Bank_of_America.Bank_of_America_Bronze.consumer_complaints;

In [0]:
-- ============================================
-- SILVER LAYER - CONSUMER COMPLAINTS
-- ============================================
-- Purpose: Individual complaint records (NO aggregation) with quality constraints and enrichments
-- Source: Bank_of_America.Bank_of_America_Bronze.consumer_complaints

-- Silver: Cleaned Consumer Complaints (one row per complaint)
CREATE TABLE IF NOT EXISTS Bank_of_America.Bank_of_America_Silver.consumer_complaints (
  complaint_id BIGINT NOT NULL,
  submitted_via STRING NOT NULL,
  date_submitted TIMESTAMP NOT NULL,
  date_received TIMESTAMP NOT NULL,
  state STRING,
  product STRING NOT NULL,
  sub_product STRING,
  issue STRING NOT NULL,
  sub_issue STRING,
  company_public_response STRING,
  company_response_to_consumer STRING,
  timely_response STRING,
  -- Derived/Enriched fields
  response_time_days INT,
  is_timely BOOLEAN,
  complaint_year INT,
  complaint_quarter INT,
  complaint_month INT,
  product_category STRING,
  resolution_status STRING,
  -- Audit field
  processed_at TIMESTAMP,
  -- Primary key constraint
  CONSTRAINT pk_consumer_complaints PRIMARY KEY(complaint_id)
)
USING DELTA
COMMENT 'Silver layer: Individual consumer complaint records with enriched fields (NO aggregation)';

SELECT 'Silver layer consumer_complaints table created successfully' as status;

In [0]:
-- ============================================
-- ETL: BRONZE TO SILVER TRANSFORMATION
-- ============================================
-- Transform consumer complaints from Bronze to Silver layer
-- Maintains individual records (NO aggregation)
-- Applies cleansing, validation, and enrichment

MERGE INTO Bank_of_America.Bank_of_America_Silver.consumer_complaints AS target
USING (
  SELECT 
    -- Core fields (cleaned and validated)
    `Complaint ID` as complaint_id,
    COALESCE(`Submitted via`, 'Unknown') as submitted_via,
    `Date submitted` as date_submitted,
    `Date received` as date_received,
    UPPER(TRIM(`State`)) as state,
    TRIM(`Product`) as product,
    TRIM(`Sub-product`) as sub_product,
    TRIM(`Issue`) as issue,
    TRIM(`Sub-issue`) as sub_issue,
    `Company public response` as company_public_response,
    `Company response to consumer` as company_response_to_consumer,
    `Timely response?` as timely_response,
    
    -- Derived fields: Response time calculation
    DATEDIFF(DAY, `Date submitted`, `Date received`) as response_time_days,
    
    -- Derived fields: Boolean conversion for timely response
    CASE 
      WHEN `Timely response?` = 'Yes' THEN TRUE
      WHEN `Timely response?` = 'No' THEN FALSE
      ELSE NULL
    END as is_timely,
    
    -- Derived fields: Date components for analysis
    YEAR(`Date submitted`) as complaint_year,
    QUARTER(`Date submitted`) as complaint_quarter,
    MONTH(`Date submitted`) as complaint_month,
    
    -- Derived fields: Standardized product categories
    CASE 
      WHEN `Product` LIKE '%Credit card%' OR `Product` LIKE '%prepaid card%' THEN 'Cards'
      WHEN `Product` LIKE '%Mortgage%' THEN 'Mortgage'
      WHEN `Product` LIKE '%Checking%' OR `Product` LIKE '%Savings%' THEN 'Deposit Accounts'
      WHEN `Product` LIKE '%Credit reporting%' THEN 'Credit Reporting'
      WHEN `Product` LIKE '%Money transfer%' THEN 'Money Services'
      WHEN `Product` LIKE '%Debt%' OR `Product` LIKE '%collection%' THEN 'Debt Collection'
      WHEN `Product` LIKE '%Student loan%' THEN 'Student Loans'
      WHEN `Product` LIKE '%Vehicle loan%' THEN 'Auto Loans'
      ELSE 'Other'
    END as product_category,
    
    -- Derived fields: Resolution status parsing
    CASE 
      WHEN `Company response to consumer` LIKE '%Closed with explanation%' THEN 'Closed - Explained'
      WHEN `Company response to consumer` LIKE '%Closed with monetary relief%' THEN 'Closed - Monetary Relief'
      WHEN `Company response to consumer` LIKE '%Closed with non-monetary relief%' THEN 'Closed - Non-Monetary Relief'
      WHEN `Company response to consumer` LIKE '%Closed without relief%' THEN 'Closed - No Relief'
      WHEN `Company response to consumer` LIKE '%In progress%' THEN 'In Progress'
      WHEN `Company response to consumer` LIKE '%Untimely response%' THEN 'Untimely Response'
      ELSE 'Unknown'
    END as resolution_status,
    
    -- Processing timestamp
    CURRENT_TIMESTAMP() as processed_at
    
  FROM Bank_of_America.Bank_of_America_Bronze.consumer_complaints
  
  -- Data quality filters: Exclude records with missing critical fields
  WHERE `Complaint ID` IS NOT NULL
    AND `Date submitted` IS NOT NULL
    AND `Date received` IS NOT NULL
    AND `Product` IS NOT NULL
    AND `Issue` IS NOT NULL
    AND `Date received` >= `Date submitted`  -- Data integrity check
    
) AS source
ON target.complaint_id = source.complaint_id

-- Update existing records if they've changed
WHEN MATCHED THEN
  UPDATE SET
    submitted_via = source.submitted_via,
    date_submitted = source.date_submitted,
    date_received = source.date_received,
    state = source.state,
    product = source.product,
    sub_product = source.sub_product,
    issue = source.issue,
    sub_issue = source.sub_issue,
    company_public_response = source.company_public_response,
    company_response_to_consumer = source.company_response_to_consumer,
    timely_response = source.timely_response,
    response_time_days = source.response_time_days,
    is_timely = source.is_timely,
    complaint_year = source.complaint_year,
    complaint_quarter = source.complaint_quarter,
    complaint_month = source.complaint_month,
    product_category = source.product_category,
    resolution_status = source.resolution_status,
    processed_at = source.processed_at

-- Insert new records
WHEN NOT MATCHED THEN
  INSERT (
    complaint_id, submitted_via, date_submitted, date_received, state,
    product, sub_product, issue, sub_issue, company_public_response,
    company_response_to_consumer, timely_response, response_time_days,
    is_timely, complaint_year, complaint_quarter, complaint_month,
    product_category, resolution_status, processed_at
  )
  VALUES (
    source.complaint_id, source.submitted_via, source.date_submitted,
    source.date_received, source.state, source.product, source.sub_product,
    source.issue, source.sub_issue, source.company_public_response,
    source.company_response_to_consumer, source.timely_response,
    source.response_time_days, source.is_timely, source.complaint_year,
    source.complaint_quarter, source.complaint_month, source.product_category,
    source.resolution_status, source.processed_at
  );

In [0]:
-- ============================================
-- SILVER LAYER VALIDATION
-- ============================================
-- Verify data quality and transformation results

-- 1. Record count comparison between Bronze and Silver
SELECT 
  'Bronze Layer' as layer,
  COUNT(*) as record_count
FROM Bank_of_America.Bank_of_America_Bronze.consumer_complaints
UNION ALL
SELECT 
  'Silver Layer' as layer,
  COUNT(*) as record_count
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints;

-- 2. Data quality summary
SELECT 
  'Total Complaints' as metric,
  COUNT(*) as value
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
UNION ALL
SELECT 
  'Timely Responses' as metric,
  SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) as value
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
UNION ALL
SELECT 
  'Average Response Time (days)' as metric,
  CAST(AVG(response_time_days) AS BIGINT) as value
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
UNION ALL
SELECT 
  'Complaints with State Info' as metric,
  SUM(CASE WHEN state IS NOT NULL THEN 1 ELSE 0 END) as value
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints;

-- 3. Sample of enriched Silver layer data
SELECT 
  complaint_id,
  submitted_via,
  date_submitted,
  state,
  product_category,
  resolution_status,
  response_time_days,
  is_timely,
  complaint_year
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
LIMIT 10;

# Data Lineage Documentation

## Consumer Complaints Data Flow

### Architecture Overview

```
Source Data --> Bronze Layer --> Silver Layer --> Gold Layer (Aggregates)
                (Raw Data)      (Individual       (Business
                                 Records)          Metrics)
```

---

## Bronze Layer: consumer_complaints

### Table Details
* **Full Name**: `Bank_of_America.Bank_of_America_Bronze.consumer_complaints`
* **Type**: Raw data ingestion table
* **Purpose**: Preserve original consumer complaint data from source systems
* **Update Pattern**: Incremental loads from source system
* **Data Retention**: Full historical data retained

### Schema (12 columns)
1. **Complaint ID** (BIGINT) - Unique identifier
2. **Submitted via** (STRING) - Submission channel
3. **Date submitted** (TIMESTAMP) - When consumer submitted
4. **Date received** (TIMESTAMP) - When company received
5. **State** (STRING) - Consumer's state
6. **Product** (STRING) - Product/service involved
7. **Sub-product** (STRING) - Detailed product type
8. **Issue** (STRING) - Main complaint issue
9. **Sub-issue** (STRING) - Detailed issue description
10. **Company public response** (STRING) - Public response text
11. **Company response to consumer** (STRING) - Resolution status
12. **Timely response?** (STRING) - Timeliness indicator

---

## Silver Layer: consumer_complaints

### Table Details
* **Full Name**: `Bank_of_America.Bank_of_America_Silver.consumer_complaints`
* **Type**: Cleaned individual records (NO aggregation)
* **Purpose**: Production-ready complaint records with enrichments
* **Update Pattern**: MERGE (upsert) from Bronze layer
* **Data Quality**: Enforces constraints and filters invalid records

### Transformation Logic

#### Data Cleansing
1. **Text Standardization**
   - TRIM() applied to all text fields
   - State codes converted to UPPERCASE
   - NULL handling with COALESCE for critical fields

2. **Data Quality Filters**
   - Exclude records with NULL complaint_id
   - Exclude records with NULL dates or product/issue
   - Validate date_received >= date_submitted
   - Ensures referential integrity

#### Data Enrichment (New Derived Fields)

3. **Response Time Analysis**
   - `response_time_days`: Calculated as DATEDIFF(date_received, date_submitted)
   - Enables performance monitoring and SLA compliance tracking

4. **Boolean Conversion**
   - `is_timely`: Converts 'Yes'/'No' string to TRUE/FALSE boolean
   - Improves query performance and analytics readability

5. **Date Dimensions**
   - `complaint_year`: YEAR(date_submitted)
   - `complaint_quarter`: QUARTER(date_submitted)
   - `complaint_month`: MONTH(date_submitted)
   - Enables time-based analysis and trending

6. **Product Categorization**
   - `product_category`: Standardized grouping of products
     * Cards (credit card, prepaid card)
     * Mortgage
     * Deposit Accounts (checking, savings)
     * Credit Reporting
     * Money Services
     * Debt Collection
     * Student Loans
     * Auto Loans
     * Other
   - Simplifies product-level analysis

7. **Resolution Status**
   - `resolution_status`: Parsed from company_response_to_consumer
     * Closed - Explained
     * Closed - Monetary Relief
     * Closed - Non-Monetary Relief
     * Closed - No Relief
     * In Progress
     * Untimely Response
     * Unknown
   - Standardizes resolution tracking

#### Data Constraints
* **Primary Key**: complaint_id (ensures uniqueness)
* **Check Constraints**:
  - submitted_via must be in valid channel list
  - date_received >= date_submitted
  - timely_response must be 'Yes', 'No', or NULL

---

## Data Flow Summary

| Stage | Layer | Record Type | Key Operations |
|-------|-------|-------------|----------------|
| 1 | Bronze | Raw individual records | Ingest from source |
| 2 | Silver | Cleaned individual records | Cleanse, validate, enrich |
| 3 | Gold | Aggregated metrics | Group by, calculate KPIs |

### Important Notes

**Silver Layer Maintains Individual Records**
* Each row = one complaint (no aggregation)
* All original fields preserved
* Additional derived fields added for analysis
* MERGE operation ensures idempotency

**Data Quality Impact**
* Bronze may contain invalid records
* Silver filters out quality issues
* Record count difference indicates rejected records
* All transformations are reversible and auditable

---

## Refresh Schedule (Recommended)

| Layer | Frequency | Dependencies |
|-------|-----------|-------------|
| Bronze | Hourly or real-time | Source system |
| Silver | Every 15 minutes | Bronze layer |
| Gold | Hourly for dashboards | Silver layer |

---

## Usage Guidelines

**When to Query Bronze**
* Auditing original source data
* Investigating data quality issues
* Debugging transformation logic

**When to Query Silver** (Recommended for most use cases)
* Standard analytical queries
* Dashboard data sources
* Data science and ML features
* Business user self-service

**When to Query Gold**
* Executive dashboards
* Pre-calculated KPI reporting
* Performance-sensitive queries
* Time-series trending analysis

---

In [0]:
-- ============================================
-- GOLD LAYER: PRODUCT PERFORMANCE METRICS
-- ============================================
-- Objective: Operational Efficiency & Business Intelligence
-- Aggregated complaints and response metrics by product category

CREATE OR REPLACE TABLE Bank_of_America.Bank_of_America_Gold.product_performance_metrics AS
SELECT 
  product_category,
  COUNT(*) as total_complaints,
  COUNT(DISTINCT state) as states_affected,
  
  -- Response Time Metrics (Operational Efficiency)
  ROUND(AVG(response_time_days), 2) as avg_response_time_days,
  CAST(PERCENTILE(response_time_days, 0.5) AS INT) as median_response_time_days,
  MAX(response_time_days) as max_response_time_days,
  CAST(PERCENTILE(response_time_days, 0.95) AS INT) as p95_response_time_days,
  
  -- Timeliness Metrics (Regulatory Compliance)
  SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) as timely_responses,
  ROUND(SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as timely_response_rate,
  
  -- Resolution Metrics (Customer Satisfaction)
  SUM(CASE WHEN resolution_status LIKE 'Closed - Monetary Relief%' THEN 1 ELSE 0 END) as monetary_relief_count,
  ROUND(SUM(CASE WHEN resolution_status LIKE 'Closed - Monetary Relief%' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as monetary_relief_rate,
  SUM(CASE WHEN resolution_status LIKE 'Closed - Non-Monetary Relief%' THEN 1 ELSE 0 END) as non_monetary_relief_count,
  SUM(CASE WHEN resolution_status LIKE 'Closed - Explained%' THEN 1 ELSE 0 END) as explained_count,
  SUM(CASE WHEN resolution_status LIKE 'Closed - No Relief%' THEN 1 ELSE 0 END) as no_relief_count,
  
  -- Date Range
  MIN(date_submitted) as earliest_complaint_date,
  MAX(date_submitted) as latest_complaint_date,
  DATEDIFF(DAY, MIN(date_submitted), MAX(date_submitted)) as days_of_data,
  
  -- Metadata
  CURRENT_TIMESTAMP() as metrics_generated_at
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
GROUP BY product_category
ORDER BY total_complaints DESC;

In [0]:
-- ============================================
-- GOLD LAYER: MONTHLY TREND METRICS
-- ============================================
-- Objective: Business Intelligence & Trend Analysis
-- Time-series analysis of complaints and performance by month

CREATE OR REPLACE TABLE Bank_of_America.Bank_of_America_Gold.monthly_trend_metrics AS
SELECT 
  complaint_year,
  complaint_month,
  DATE_TRUNC('MONTH', date_submitted) as month_start_date,
  
  -- Volume Metrics
  COUNT(*) as total_complaints,
  COUNT(DISTINCT product_category) as distinct_products,
  COUNT(DISTINCT state) as distinct_states,
  COUNT(DISTINCT submitted_via) as distinct_channels,
  
  -- Performance Metrics
  ROUND(AVG(response_time_days), 2) as avg_response_time_days,
  ROUND(SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as timely_response_rate,
  
  -- Resolution Distribution
  SUM(CASE WHEN resolution_status LIKE 'Closed - Monetary Relief%' THEN 1 ELSE 0 END) as monetary_relief_count,
  ROUND(SUM(CASE WHEN resolution_status LIKE 'Closed - Monetary Relief%' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as monetary_relief_rate,
  SUM(CASE WHEN resolution_status LIKE 'Closed%' THEN 1 ELSE 0 END) as closed_complaints,
  ROUND(SUM(CASE WHEN resolution_status LIKE 'Closed%' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as closure_rate,
  
  -- Metadata
  CURRENT_TIMESTAMP() as metrics_generated_at
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
GROUP BY complaint_year, complaint_month, DATE_TRUNC('MONTH', date_submitted)
ORDER BY complaint_year DESC, complaint_month DESC;

In [0]:
-- ============================================
-- GOLD LAYER: STATE PERFORMANCE METRICS
-- ============================================
-- Objective: Operational Efficiency & Geographic Analysis
-- Complaint volume and resolution performance by state

CREATE OR REPLACE TABLE Bank_of_America.Bank_of_America_Gold.state_performance_metrics AS
SELECT 
  state,
  COUNT(*) as total_complaints,
  COUNT(DISTINCT product_category) as distinct_products_affected,
  
  -- Response Time Performance
  ROUND(AVG(response_time_days), 2) as avg_response_time_days,
  CAST(PERCENTILE(response_time_days, 0.5) AS INT) as median_response_time_days,
  
  -- Compliance Metrics
  ROUND(SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as timely_response_rate,
  
  -- Resolution Metrics
  ROUND(SUM(CASE WHEN resolution_status LIKE 'Closed - Monetary Relief%' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as monetary_relief_rate,
  ROUND(SUM(CASE WHEN resolution_status LIKE 'Closed%' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as resolution_rate,
  
  -- Date Range
  MIN(date_submitted) as earliest_complaint_date,
  MAX(date_submitted) as latest_complaint_date,
  
  -- Metadata
  CURRENT_TIMESTAMP() as metrics_generated_at
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
WHERE state IS NOT NULL
GROUP BY state
ORDER BY total_complaints DESC;

In [0]:
-- ============================================
-- GOLD LAYER: RESOLUTION PERFORMANCE SUMMARY
-- ============================================
-- Objective: Regulatory Compliance & Executive Reporting
-- Overall complaint resolution performance by status

CREATE OR REPLACE TABLE Bank_of_America.Bank_of_America_Gold.resolution_performance_summary AS
SELECT 
  resolution_status,
  COUNT(*) as complaint_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage_of_total,
  
  -- Response Time Analysis
  ROUND(AVG(response_time_days), 2) as avg_response_time_days,
  CAST(PERCENTILE(response_time_days, 0.5) AS INT) as median_response_time_days,
  CAST(PERCENTILE(response_time_days, 0.75) AS INT) as p75_response_time_days,
  CAST(PERCENTILE(response_time_days, 0.95) AS INT) as p95_response_time_days,
  MAX(response_time_days) as max_response_time_days,
  
  -- Timeliness Compliance
  SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) as timely_count,
  ROUND(SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as timely_rate,
  
  -- Coverage
  COUNT(DISTINCT product_category) as products_affected,
  COUNT(DISTINCT state) as states_affected,
  COUNT(DISTINCT submitted_via) as channels_used,
  
  -- Metadata
  CURRENT_TIMESTAMP() as metrics_generated_at
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
GROUP BY resolution_status
ORDER BY complaint_count DESC;

In [0]:
-- ============================================
-- GOLD LAYER: CHANNEL PERFORMANCE METRICS
-- ============================================
-- Objective: Operational Efficiency & Customer Experience
-- Complaint volume and resolution performance by submission channel

CREATE OR REPLACE TABLE Bank_of_America.Bank_of_America_Gold.channel_performance_metrics AS
SELECT 
  submitted_via as submission_channel,
  COUNT(*) as total_complaints,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as percentage_of_total,
  
  -- Response Time Performance
  ROUND(AVG(response_time_days), 2) as avg_response_time_days,
  CAST(PERCENTILE(response_time_days, 0.5) AS INT) as median_response_time_days,
  CAST(PERCENTILE(response_time_days, 0.95) AS INT) as p95_response_time_days,
  
  -- Timeliness Metrics
  SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) as timely_responses,
  ROUND(SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as timely_response_rate,
  
  -- Resolution Quality
  ROUND(SUM(CASE WHEN resolution_status LIKE 'Closed - Monetary Relief%' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as monetary_relief_rate,
  ROUND(SUM(CASE WHEN resolution_status LIKE 'Closed%' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as closure_rate,
  
  -- Coverage
  COUNT(DISTINCT product_category) as products_handled,
  COUNT(DISTINCT state) as states_served,
  
  -- Metadata
  CURRENT_TIMESTAMP() as metrics_generated_at
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
GROUP BY submitted_via
ORDER BY total_complaints DESC;

In [0]:
-- ============================================
-- GOLD LAYER: ISSUE CATEGORY ANALYSIS
-- ============================================
-- Objective: Customer Insights & Risk Assessment
-- Most common complaint issues and their resolution performance

CREATE OR REPLACE TABLE Bank_of_America.Bank_of_America_Gold.issue_category_analysis AS
SELECT 
  issue as complaint_issue,
  product_category,
  COUNT(*) as complaint_count,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY product_category), 2) as pct_of_product_complaints,
  
  -- Response Performance
  ROUND(AVG(response_time_days), 2) as avg_response_time_days,
  ROUND(SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as timely_response_rate,
  
  -- Resolution Outcomes
  SUM(CASE WHEN resolution_status LIKE 'Closed - Monetary Relief%' THEN 1 ELSE 0 END) as monetary_relief_count,
  SUM(CASE WHEN resolution_status LIKE 'Closed - Non-Monetary Relief%' THEN 1 ELSE 0 END) as non_monetary_relief_count,
  SUM(CASE WHEN resolution_status LIKE 'Closed - No Relief%' THEN 1 ELSE 0 END) as no_relief_count,
  ROUND(SUM(CASE WHEN resolution_status LIKE 'Closed%' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as resolution_rate,
  
  -- Geographic Distribution
  COUNT(DISTINCT state) as states_affected,
  
  -- Metadata
  CURRENT_TIMESTAMP() as metrics_generated_at
FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
GROUP BY issue, product_category
HAVING COUNT(*) >= 10  -- Filter for statistically meaningful issues
ORDER BY product_category, complaint_count DESC;

In [0]:
-- ============================================
-- GOLD LAYER VALIDATION
-- ============================================
-- Verify all Gold layer tables and preview key business metrics

-- 1. List all Gold layer tables
SHOW TABLES IN Bank_of_America.Bank_of_America_Gold;

-- 2. Product Performance Overview (Top 5 Products by Complaint Volume)
SELECT 
  product_category,
  total_complaints,
  avg_response_time_days,
  timely_response_rate,
  monetary_relief_rate
FROM Bank_of_America.Bank_of_America_Gold.product_performance_metrics
ORDER BY total_complaints DESC
LIMIT 5;

-- 3. Recent Monthly Trends (Last 6 Months)
SELECT 
  complaint_year,
  complaint_month,
  total_complaints,
  avg_response_time_days,
  timely_response_rate,
  closure_rate
FROM Bank_of_America.Bank_of_America_Gold.monthly_trend_metrics
ORDER BY complaint_year DESC, complaint_month DESC
LIMIT 6;

-- 4. Top 5 States by Complaint Volume
SELECT 
  state,
  total_complaints,
  timely_response_rate,
  resolution_rate
FROM Bank_of_America.Bank_of_America_Gold.state_performance_metrics
ORDER BY total_complaints DESC
LIMIT 5;

-- 5. Resolution Performance Summary
SELECT 
  resolution_status,
  complaint_count,
  percentage_of_total,
  avg_response_time_days,
  timely_rate
FROM Bank_of_America.Bank_of_America_Gold.resolution_performance_summary
ORDER BY complaint_count DESC;

In [0]:
-- ============================================
-- GOLD LAYER COMPLETE OVERVIEW
-- ============================================
-- Summary of all Gold layer tables with record counts

SELECT 'product_performance_metrics' as table_name,
       COUNT(*) as record_count,
       'Product-level KPIs: complaints, response times, resolution rates' as description
FROM Bank_of_America.Bank_of_America_Gold.product_performance_metrics

UNION ALL

SELECT 'monthly_trend_metrics' as table_name,
       COUNT(*) as record_count,
       'Time-series trends: monthly complaint volumes and performance' as description
FROM Bank_of_America.Bank_of_America_Gold.monthly_trend_metrics

UNION ALL

SELECT 'state_performance_metrics' as table_name,
       COUNT(*) as record_count,
       'Geographic analysis: state-level complaint volumes and outcomes' as description
FROM Bank_of_America.Bank_of_America_Gold.state_performance_metrics

UNION ALL

SELECT 'resolution_performance_summary' as table_name,
       COUNT(*) as record_count,
       'Resolution status distribution with compliance metrics' as description
FROM Bank_of_America.Bank_of_America_Gold.resolution_performance_summary

UNION ALL

SELECT 'channel_performance_metrics' as table_name,
       COUNT(*) as record_count,
       'Submission channel analysis: Web, Phone, Referral performance' as description
FROM Bank_of_America.Bank_of_America_Gold.channel_performance_metrics

UNION ALL

SELECT 'issue_category_analysis' as table_name,
       COUNT(*) as record_count,
       'Issue-level insights: most common problems by product' as description
FROM Bank_of_America.Bank_of_America_Gold.issue_category_analysis

ORDER BY table_name;

# Gold Layer Documentation

## Overview

The Gold layer contains **6 aggregated business-level tables** that transform the Silver layer's individual complaint records into actionable KPIs and metrics. These tables are designed to support the project's key objectives.

---

## Gold Layer Tables Aligned with Project Objectives

### 1. Product Performance Metrics
**Table**: `Bank_of_America.Bank_of_America_Gold.product_performance_metrics`  
**Records**: 9 product categories  
**Objectives Supported**:
* ✅ **Operational Efficiency** - Identify products with highest complaint volumes and longest response times
* ✅ **Business Intelligence** - Track product-level KPIs for executive dashboards
* ✅ **Customer Insights** - Understand which products require service improvements

**Key Metrics**:
* Total complaints by product category
* Average/median/P95 response times
* Timely response rate (regulatory compliance)
* Resolution outcome distribution (monetary relief, non-monetary, explained, no relief)
* Geographic coverage (states affected)

**Business Use Cases**:
* Product managers tracking service quality
* Executive dashboards showing product performance
* Resource allocation decisions based on complaint volumes

---

### 2. Monthly Trend Metrics
**Table**: `Bank_of_America.Bank_of_America_Gold.monthly_trend_metrics`  
**Records**: 76 months of data  
**Objectives Supported**:
* ✅ **Business Intelligence** - Time-series analysis for strategic planning
* ✅ **Operational Efficiency** - Track performance trends over time
* ✅ **Risk Assessment** - Identify seasonal patterns or emerging issues

**Key Metrics**:
* Monthly complaint volumes and growth rates
* Response time trends
* Timeliness compliance trends
* Resolution rate trends
* Product and channel diversity by month

**Business Use Cases**:
* Monthly/quarterly executive reports
* Forecasting future complaint volumes
* Measuring impact of service improvements
* Seasonal trend analysis

---

### 3. State Performance Metrics
**Table**: `Bank_of_America.Bank_of_America_Gold.state_performance_metrics`  
**Records**: 51 states/territories  
**Objectives Supported**:
* ✅ **Operational Efficiency** - Geographic resource allocation
* ✅ **Business Intelligence** - Regional performance comparison
* ✅ **Customer Insights** - State-specific service quality

**Key Metrics**:
* Complaint volumes by state
* Response time performance by geography
* Timeliness and resolution rates by state
* Product diversity affected per state

**Business Use Cases**:
* Regional manager performance tracking
* Geographic resource allocation (call centers, branches)
* State-specific compliance reporting
* Targeted improvement initiatives

---

### 4. Resolution Performance Summary
**Table**: `Bank_of_America.Bank_of_America_Gold.resolution_performance_summary`  
**Records**: 5 resolution status categories  
**Objectives Supported**:
* ✅ **Regulatory Compliance** - Track resolution outcomes for regulatory reporting
* ✅ **Operational Efficiency** - Monitor case resolution effectiveness
* ✅ **Customer Insights** - Understand resolution quality

**Key Metrics**:
* Distribution across resolution statuses (Closed - Explained: 65.7%, Monetary Relief: 23.5%, etc.)
* Response time by resolution type
* Timeliness compliance by outcome
* Coverage metrics (products, states, channels affected)

**Business Use Cases**:
* Regulatory compliance reporting (CFPB, OCC)
* Executive KPI dashboards
* Customer satisfaction correlation analysis
* Process improvement initiatives

---

### 5. Channel Performance Metrics
**Table**: `Bank_of_America.Bank_of_America_Gold.channel_performance_metrics`  
**Records**: 7 submission channels (Web, Phone, Referral, etc.)  
**Objectives Supported**:
* ✅ **Operational Efficiency** - Optimize channel performance
* ✅ **Customer Insights** - Understand customer channel preferences
* ✅ **Business Intelligence** - Channel strategy decisions

**Key Metrics**:
* Complaint volume by channel
* Response time performance by channel
* Timeliness and resolution rates by channel
* Product and geographic coverage per channel

**Business Use Cases**:
* Digital transformation ROI analysis
* Call center vs. web portal performance
* Channel optimization and investment decisions
* Customer experience improvement by channel

---

### 6. Issue Category Analysis
**Table**: `Bank_of_America.Bank_of_America_Gold.issue_category_analysis`  
**Records**: 79 issue-product combinations (filtered for statistical significance)  
**Objectives Supported**:
* ✅ **Customer Insights** - Identify most common customer pain points
* ✅ **Risk Assessment** - Track recurring issues that may indicate systemic problems
* ✅ **Operational Efficiency** - Prioritize issue resolution efforts

**Key Metrics**:
* Complaint volumes by issue category and product
* Percentage of product complaints per issue
* Response time and timeliness by issue type
* Resolution outcome distribution by issue
* Geographic spread of issues

**Business Use Cases**:
* Root cause analysis for product improvements
* Training program development for service reps
* Proactive issue mitigation strategies
* Product design and UX improvements

---

## Medallion Architecture Complete

| Layer | Purpose | Record Type | Table Count | Total Records |
|-------|---------|-------------|-------------|---------------|
| **Bronze** | Raw data preservation | Individual | 2 | 62,516 |
| **Silver** | Cleaned production data | Individual | 1 | 62,516 |
| **Gold** | Business aggregates | Aggregated | 6 | 227 metrics |

### Data Flow Summary

```
Bronze (Raw)           Silver (Cleaned)          Gold (Business Metrics)
62,516 complaints  →   62,516 enriched      →    227 aggregated KPIs
                       individual records         across 6 dimensions:
                       + 8 derived fields         - Products (9)
                                                   - Time (76 months)
                                                   - Geography (51 states)
                                                   - Resolution (5 statuses)
                                                   - Channels (7)
                                                   - Issues (79)
```

---

## Project Objectives Achievement

### ✅ Customer Insights & Segmentation
* Issue category analysis identifies customer pain points
* Product and channel preferences tracked
* Geographic complaint patterns analyzed

### ✅ Operational Efficiency
* Product, channel, and state performance metrics enable targeted improvements
* Response time tracking across all dimensions
* Resource allocation guided by data

### ✅ Risk Assessment
* Issue trending identifies emerging problems
* Monthly metrics detect anomalies
* Resolution outcome monitoring

### ✅ Regulatory Compliance
* Timeliness metrics track SLA compliance (93.8% timely response rate)
* Resolution performance documented for regulatory reporting
* Audit trail maintained through all layers

### ✅ Business Intelligence & Reporting
* Executive-ready KPI tables for dashboards
* Time-series data for trend analysis
* Multi-dimensional analysis (product × geography × channel × issue)

---

## Usage Recommendations

### Dashboard Development
* **Executive Dashboard**: Use product_performance_metrics + monthly_trend_metrics + resolution_performance_summary
* **Operational Dashboard**: Use channel_performance_metrics + state_performance_metrics
* **Deep-Dive Analysis**: Use issue_category_analysis for root cause investigations

### Refresh Strategy
* **Bronze Layer**: Incremental loads as new data arrives
* **Silver Layer**: Run ETL hourly or on-demand
* **Gold Layer**: Refresh hourly (CTAS queries are fast on aggregated data)

### Query Performance
* Gold layer queries are optimized for dashboard performance
* Pre-aggregated metrics eliminate need for complex joins
* Small row counts (5-79 records per table) enable sub-second query times

---

## Next Steps

1. **Create Dashboards**: Build Lakeview dashboards using Gold layer tables
2. **Schedule Refreshes**: Automate ETL pipeline (Bronze → Silver → Gold)
3. **Add Alerts**: Set up alerts for KPI thresholds (e.g., timeliness < 90%)
4. **Expand Analysis**: Add year-over-year comparisons, cohort analysis
5. **ML Integration**: Use Silver layer for predictive models (resolution time prediction, issue classification)

In [0]:
-- ============================================
-- EXPANDED ANALYSIS: YEAR-OVER-YEAR COMPARISON
-- ============================================
-- Compare complaint volumes and performance metrics across years

WITH yearly_metrics AS (
  SELECT 
    complaint_year,
    COUNT(*) as total_complaints,
    ROUND(AVG(response_time_days), 2) as avg_response_time,
    ROUND(SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as timely_rate,
    ROUND(SUM(CASE WHEN resolution_status LIKE 'Closed - Monetary Relief%' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as monetary_relief_rate,
    COUNT(DISTINCT product_category) as products_affected,
    COUNT(DISTINCT state) as states_affected
  FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
  GROUP BY complaint_year
),
yoy_changes AS (
  SELECT 
    complaint_year,
    total_complaints,
    total_complaints - LAG(total_complaints) OVER (ORDER BY complaint_year) as yoy_complaint_change,
    ROUND((total_complaints - LAG(total_complaints) OVER (ORDER BY complaint_year)) * 100.0 / 
          NULLIF(LAG(total_complaints) OVER (ORDER BY complaint_year), 0), 2) as yoy_complaint_pct_change,
    avg_response_time,
    avg_response_time - LAG(avg_response_time) OVER (ORDER BY complaint_year) as yoy_response_time_change,
    timely_rate,
    timely_rate - LAG(timely_rate) OVER (ORDER BY complaint_year) as yoy_timely_rate_change,
    monetary_relief_rate,
    products_affected,
    states_affected
  FROM yearly_metrics
)
SELECT 
  complaint_year,
  total_complaints,
  yoy_complaint_change,
  yoy_complaint_pct_change,
  avg_response_time,
  yoy_response_time_change,
  timely_rate,
  yoy_timely_rate_change,
  monetary_relief_rate,
  products_affected,
  states_affected
FROM yoy_changes
ORDER BY complaint_year DESC;

In [0]:
-- ============================================
-- EXPANDED ANALYSIS: PRODUCT COHORT ANALYSIS
-- ============================================
-- Analyze how products perform over time (cohort = year first complaint seen)

WITH product_cohorts AS (
  SELECT 
    product_category,
    complaint_year,
    MIN(complaint_year) OVER (PARTITION BY product_category) as cohort_year,
    COUNT(*) as complaints_in_year,
    ROUND(AVG(response_time_days), 2) as avg_response_time,
    ROUND(SUM(CASE WHEN is_timely = TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as timely_rate
  FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
  GROUP BY product_category, complaint_year
)
SELECT 
  product_category,
  cohort_year,
  complaint_year,
  complaint_year - cohort_year as years_since_cohort,
  complaints_in_year,
  avg_response_time,
  timely_rate,
  SUM(complaints_in_year) OVER (PARTITION BY product_category ORDER BY complaint_year) as cumulative_complaints
FROM product_cohorts
WHERE complaint_year >= 2020  -- Focus on recent years
ORDER BY product_category, complaint_year DESC;

In [0]:
-- ============================================
-- EXPANDED ANALYSIS: SEASONALITY PATTERNS
-- ============================================
-- Identify seasonal patterns by month across all years

WITH monthly_patterns AS (
  SELECT 
    complaint_month,
    complaint_year,
    COUNT(*) as monthly_complaints,
    ROUND(AVG(response_time_days), 2) as avg_response_time
  FROM Bank_of_America.Bank_of_America_Silver.consumer_complaints
  GROUP BY complaint_month, complaint_year
)
SELECT 
  complaint_month,
  CASE complaint_month
    WHEN 1 THEN 'January'
    WHEN 2 THEN 'February'
    WHEN 3 THEN 'March'
    WHEN 4 THEN 'April'
    WHEN 5 THEN 'May'
    WHEN 6 THEN 'June'
    WHEN 7 THEN 'July'
    WHEN 8 THEN 'August'
    WHEN 9 THEN 'September'
    WHEN 10 THEN 'October'
    WHEN 11 THEN 'November'
    WHEN 12 THEN 'December'
  END as month_name,
  ROUND(AVG(monthly_complaints), 0) as avg_monthly_complaints,
  MIN(monthly_complaints) as min_complaints,
  MAX(monthly_complaints) as max_complaints,
  CAST(PERCENTILE(monthly_complaints, 0.5) AS INT) as median_complaints,
  ROUND(AVG(avg_response_time), 2) as avg_response_time_across_years,
  COUNT(DISTINCT complaint_year) as years_of_data
FROM monthly_patterns
GROUP BY complaint_month
ORDER BY complaint_month;

In [0]:
-- ============================================
-- ALERT MONITORING: KPI THRESHOLD CHECKS
-- ============================================
-- Identify products/states/channels falling below acceptable thresholds

-- Threshold: Timely Response Rate should be >= 90%
-- Threshold: Average Response Time should be <= 2 days
-- Threshold: Resolution Rate should be >= 95%

SELECT 
  'Product Performance' as alert_category,
  product_category as entity,
  CONCAT('Timely Rate: ', timely_response_rate, '%') as metric_value,
  'CRITICAL: Below 90% threshold' as alert_severity,
  CONCAT('Product ', product_category, ' has timely response rate of ', timely_response_rate, '%, below 90% threshold') as alert_message,
  CURRENT_TIMESTAMP() as alert_timestamp
FROM Bank_of_America.Bank_of_America_Gold.product_performance_metrics
WHERE timely_response_rate < 90.0

UNION ALL

SELECT 
  'Product Performance' as alert_category,
  product_category as entity,
  CONCAT('Avg Response Time: ', avg_response_time_days, ' days') as metric_value,
  'WARNING: Above 2-day threshold' as alert_severity,
  CONCAT('Product ', product_category, ' has average response time of ', avg_response_time_days, ' days, above 2-day threshold') as alert_message,
  CURRENT_TIMESTAMP() as alert_timestamp
FROM Bank_of_America.Bank_of_America_Gold.product_performance_metrics
WHERE avg_response_time_days > 2.0

UNION ALL

SELECT 
  'State Performance' as alert_category,
  state as entity,
  CONCAT('Timely Rate: ', timely_response_rate, '%') as metric_value,
  'CRITICAL: Below 90% threshold' as alert_severity,
  CONCAT('State ', state, ' has timely response rate of ', timely_response_rate, '%, below 90% threshold') as alert_message,
  CURRENT_TIMESTAMP() as alert_timestamp
FROM Bank_of_America.Bank_of_America_Gold.state_performance_metrics
WHERE timely_response_rate < 90.0
  AND total_complaints >= 100  -- Only alert for states with significant volume

UNION ALL

SELECT 
  'Channel Performance' as alert_category,
  submission_channel as entity,
  CONCAT('Timely Rate: ', timely_response_rate, '%') as metric_value,
  'WARNING: Below 90% threshold' as alert_severity,
  CONCAT('Channel ', submission_channel, ' has timely response rate of ', timely_response_rate, '%, below 90% threshold') as alert_message,
  CURRENT_TIMESTAMP() as alert_timestamp
FROM Bank_of_America.Bank_of_America_Gold.channel_performance_metrics
WHERE timely_response_rate < 90.0

ORDER BY alert_severity DESC, alert_category, entity;